# Food Delivery Time Prediction
**ACM AI/ML Hackathon - PS1** | Target: `delivery_time_min` | Dataset: `food_delivery_time.csv` (1,000 rows, 9 columns)

### 40-minute clock
| Min | Section | Goal |
|---|---|---|
| 0-4 | 1. Load | Data in, shape and dtypes confirmed |
| 4-12 | 2-3. Check + EDA | Missing values, outliers, correlations |
| 12-18 | 4. Prepare + split | Drop junk columns, 80/20 split |
| 18-30 | 5-6. Models | Baseline, linear regression, 2 stronger models |
| 30-36 | 7. Evaluate | Test-set MAE / RMSE / R2, residuals, feature importance |
| 36-40 | 8-9. Explain | Deployment-time check + written justification |

**Rule:** have a working baseline by minute 18. Skip anything marked *(bonus)* if you are behind.

**Approach:** split the data -> guess the average (the number to beat) -> linear regression (maths) -> tree ensembles -> test once on unseen orders.

## 1. Load the data (0-4 min)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

BLUE, ORANGE, GREEN, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8985"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e1", "axes.axisbelow": True})
TARGET = "delivery_time_min"
SEED = 42

In [ ]:
# Looks for the file in the usual places; if not found, asks you to upload it.
candidates = ["food_delivery_time.csv", "/content/food_delivery_time.csv",
              "/content/drive/MyDrive/food_delivery_time.csv"]
path = next((p for p in candidates if os.path.exists(p)), None)
if path is None:
    from google.colab import files
    uploaded = files.upload()          # pick food_delivery_time.csv
    path = list(uploaded.keys())[0]

raw = pd.read_csv(path)
print("Rows, columns:", raw.shape)
raw.head()

## 2. Check the data (4-8 min)
Before modelling: nothing missing, no duplicate orders, no impossible values (negative times or distances).

In [ ]:
print("Missing values  :", raw.isna().sum().sum())
print("Duplicate rows  :", raw.duplicated().sum())
print("Duplicate IDs   :", raw["order_id"].duplicated().sum())
num = raw.drop(columns="order_id")
print("Negative values :", (num < 0).sum().sum())
raw.info()
raw.describe().T.round(2)

In [ ]:
# Are distance_km and distance_miles the same information in two units?
ratio = raw["distance_miles"] / raw["distance_km"]
print("Correlation km vs miles:", round(raw["distance_km"].corr(raw["distance_miles"]), 6))
print("miles / km ratio  min, max:", round(ratio.min(), 4), round(ratio.max(), 4), "(1 km = 0.6214 miles)")

If the correlation is 1.0 and the ratio is constant, `distance_miles` is just `distance_km` in other units. It adds no information and breaks the maths of linear regression (the normal equation needs `X'X` to be invertible, and two perfectly dependent columns make it singular). **We keep one and drop the other.** `order_id` is only a label, so it goes too.

## 3. Exploratory data analysis (8-12 min)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(raw[TARGET], bins=30, color=BLUE)
ax[0].set_xlabel("Delivery time (min)"); ax[0].set_ylabel("Orders"); ax[0].set_title("Target distribution")
ax[1].boxplot(raw[TARGET], vert=False)
ax[1].set_xlabel("Delivery time (min)"); ax[1].set_title("Outlier check")
plt.show()
print(f"Mean {raw[TARGET].mean():.1f} min | median {raw[TARGET].median():.1f} | std {raw[TARGET].std():.1f} | skew {raw[TARGET].skew():.2f}")

In [ ]:
corr = raw.drop(columns=["order_id", TARGET]).corrwith(raw[TARGET]).sort_values()
plt.figure(figsize=(7, 3.6))
plt.barh(corr.index, corr.values, color=[ORANGE if v < 0 else BLUE for v in corr.values])
for i, v in enumerate(corr.values):
    plt.text(v + (0.02 if v >= 0 else -0.02), i, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right")
plt.xlim(-1, 1); plt.axvline(0, color=GREY, linewidth=1)
plt.xlabel("Correlation with delivery time"); plt.title("Which columns move with delivery time?")
plt.show()

In [ ]:
cols = ["distance_km", "prep_time_min", "traffic_delay_min", "courier_wait_min"]
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
for a, c in zip(ax, cols):
    a.scatter(raw[c], raw[TARGET], s=8, alpha=0.4, color=BLUE)
    a.set_xlabel(c); a.set_ylabel(TARGET)
plt.tight_layout(); plt.show()

**Read the plots:** which columns have a clear straight-line pattern with the target? Are there points far from the rest? Note 1-2 findings here for your write-up.

## 4. Prepare + train/test split (12-18 min)
We hide 20% of the orders as a **test set**. The model learns from the other 80% and the test set tells us how it does on orders it has never seen. Fixed seed -> same split every run.

In [ ]:
data = raw.drop(columns=["order_id", "distance_miles"]).drop_duplicates()

# Fill any gaps with the median (only matters if NaNs exist).
# Strictly the median should come from the training set only; with 0 NaNs this changes nothing.
data = data.fillna(data.median())

# Outliers: we KEEP them unless clearly impossible. Check how many fall far outside the IQR range.
q1, q3 = data[TARGET].quantile([0.25, 0.75]); iqr = q3 - q1
n_out = ((data[TARGET] < q1 - 3 * iqr) | (data[TARGET] > q3 + 3 * iqr)).sum()
print("Extreme target outliers (3 x IQR):", n_out)

FEATURES = [c for c in data.columns if c != TARGET]
X, y = data[FEATURES], data[TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Features:", FEATURES)
print("Training orders:", len(X_tr), " Test orders:", len(X_te))

## 5. Baseline + linear regression (18-24 min)

**The model:** `y_hat = b + m1*x1 + m2*x2 + ... + mn*xn`. Linear regression picks the slopes that minimise the mean squared error
`MSE = (1/n) * sum (y_i - y_hat_i)^2`, which has the closed-form **normal equation** `beta = (X'X)^-1 X'y`.

**Metrics:** MAE = average error in minutes. RMSE = like MAE but big misses count more. R2 = share of the differences between orders that the model explains (1 = perfect, 0 = no better than guessing the average).

In [ ]:
def evaluate(actual, predicted):
    actual, predicted = np.asarray(actual), np.asarray(predicted)
    mae = np.mean(np.abs(actual - predicted))
    rmse = np.sqrt(np.mean((actual - predicted) ** 2))
    r2 = 1 - np.sum((actual - predicted) ** 2) / np.sum((actual - actual.mean()) ** 2)
    return mae, rmse, r2

# Baseline: always guess the training average. Every real model must beat this.
base_pred = np.full(len(y_te), y_tr.mean())
mae, rmse, r2 = evaluate(y_te, base_pred)
print(f"Guess the average   MAE {mae:6.2f}  RMSE {rmse:6.2f}  R2 {r2:6.3f}")

In [ ]:
lin = LinearRegression().fit(X_tr, y_tr)

# The same line by hand, with the normal equation (lstsq = a numerically safe version of the inverse)
A = np.column_stack([np.ones(len(X_tr)), X_tr.to_numpy()])
beta = np.linalg.lstsq(A, y_tr.to_numpy(), rcond=None)[0]

print("             intercept " + " ".join(f"{c[:12]:>12s}" for c in FEATURES))
print("Normal eq.  ", " ".join(f"{v:12.3f}" for v in beta))
print("sklearn     ", " ".join(f"{v:12.3f}" for v in np.r_[lin.intercept_, lin.coef_]))

mae, rmse, r2 = evaluate(y_te, lin.predict(X_te))
print(f"\nLinear regression   MAE {mae:6.2f}  RMSE {rmse:6.2f}  R2 {r2:6.3f}")

**How to read the slopes:** each number is the change in predicted delivery time (minutes) for +1 of that column, with all other columns unchanged. Write down the 2 biggest ones for your explanation.

### Gradient descent *(bonus - skip if behind)*
Same line, found by walking downhill: `m <- m - lr * dMSE/dm`. Features are scaled first so every slope learns at a similar speed.

In [ ]:
mu, sd = X_tr.mean(), X_tr.std()
Z = ((X_tr - mu) / sd).to_numpy()
yy = y_tr.to_numpy()
w, b0, lr = np.zeros(Z.shape[1]), 0.0, 0.1
losses = []
for step in range(300):
    err = yy - (Z @ w + b0)
    losses.append(np.mean(err ** 2))
    w  += lr * 2 * np.mean(Z * err[:, None], axis=0)
    b0 += lr * 2 * np.mean(err)

ref = LinearRegression().fit(Z, yy)
print("Biggest difference vs LinearRegression (scaled slopes):", np.abs(w - ref.coef_).max().round(6))
plt.figure(figsize=(7, 3)); plt.plot(losses, color=BLUE); plt.yscale("log")
plt.xlabel("Step"); plt.ylabel("MSE"); plt.title("Gradient descent converging"); plt.show()

## 6. Compare models with cross-validation (24-30 min)
We pick the winner using **only the training set** (5-fold CV on MAE), so the test set stays untouched until the end. Linear models need scaled inputs, so scaling lives inside a `Pipeline` and is fitted on the training folds only (no leakage).

In [ ]:
models = {
    "Linear":       make_pipeline(StandardScaler(), LinearRegression()),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    "GradBoost":    GradientBoostingRegressor(random_state=SEED),
}
cv_mae = {}
for name, mdl in models.items():
    scores = -cross_val_score(mdl, X_tr, y_tr, cv=5, scoring="neg_mean_absolute_error")
    cv_mae[name] = scores.mean()
    print(f"{name:13s} CV MAE {scores.mean():6.2f}  (+/- {scores.std():.2f})")

best_name = min(cv_mae, key=cv_mae.get)
print("\nBest by CV MAE:", best_name)

If **Linear** wins or ties, say so: the problem is mostly additive (delivery time ~ prep + travel + delay + wait), so the simpler, more explainable model is the better choice. Only if a tree model wins clearly is the extra complexity justified.

*(Optional, only if you have 3+ minutes: `GridSearchCV` on 1-2 parameters of the winner, e.g. `max_depth` and `n_estimators`.)*

## 7. Final evaluation on the test set (30-36 min)
The test set is used **once**, with the chosen model.

In [ ]:
best = clone(models[best_name]).fit(X_tr, y_tr)
pred = best.predict(X_te)
mae, rmse, r2 = evaluate(y_te, pred)
print(f"{best_name} on UNSEEN test orders:  MAE {mae:.2f} min   RMSE {rmse:.2f} min   R2 {r2:.3f}")

# Compare all three on the test set for the write-up table
test_table = {}
for name, mdl in models.items():
    p = clone(mdl).fit(X_tr, y_tr).predict(X_te)
    test_table[name] = evaluate(y_te, p)
test_table["Guess the average"] = evaluate(y_te, base_pred)
pd.DataFrame(test_table, index=["MAE", "RMSE", "R2"]).T.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
lo, hi = min(y_te.min(), pred.min()), max(y_te.max(), pred.max())
ax[0].scatter(y_te, pred, s=14, alpha=0.5, color=BLUE)
ax[0].plot([lo, hi], [lo, hi], color=ORANGE, linewidth=2, label="Perfect prediction")
ax[0].set_xlabel("Actual delivery time (min)"); ax[0].set_ylabel("Predicted (min)")
ax[0].set_title("Predicted vs actual (test set)"); ax[0].legend()

resid = y_te.to_numpy() - pred
ax[1].scatter(pred, resid, s=14, alpha=0.5, color=BLUE)
ax[1].axhline(0, color=ORANGE, linewidth=2)
ax[1].set_xlabel("Predicted (min)"); ax[1].set_ylabel("Residual = actual - predicted")
ax[1].set_title("Residuals: should be a flat cloud around 0")
plt.tight_layout(); plt.show()
print(f"Within 5 min: {np.mean(np.abs(resid) <= 5):.0%} of orders | within 10 min: {np.mean(np.abs(resid) <= 10):.0%}")

In [ ]:
imp = permutation_importance(best, X_te, y_te, n_repeats=10, random_state=SEED, scoring="neg_mean_absolute_error")
order = np.argsort(imp.importances_mean)
plt.figure(figsize=(7, 3.6))
plt.barh(np.array(FEATURES)[order], imp.importances_mean[order], color=BLUE)
plt.xlabel("Increase in MAE (min) when the column is shuffled")
plt.title("What drives the prediction?"); plt.show()

## 8. Deployment-time check: leakage awareness (36-38 min)
The task is to predict **from the moment an order is placed**. At that moment, do we already know the real `traffic_delay_min` and `courier_wait_min`? Probably not: they are things that happen *after* the order. Using them is a mild form of **data leakage** (information the model would not have in real use), so a test score that includes them can look better than real-world performance.

We measure the difference by retraining without them.

In [ ]:
AT_ORDER = [c for c in FEATURES if c not in ("traffic_delay_min", "courier_wait_min")]
m2 = clone(models[best_name]).fit(X_tr[AT_ORDER], y_tr)
mae2, rmse2, r22 = evaluate(y_te, m2.predict(X_te[AT_ORDER]))
print("Features known at order time:", AT_ORDER)
print(f"All features        MAE {mae:5.2f}  RMSE {rmse:5.2f}  R2 {r2:.3f}")
print(f"Order-time only     MAE {mae2:5.2f}  RMSE {rmse2:5.2f}  R2 {r22:.3f}")

## 9. Predict one order (try it)

In [ ]:
my_order = {c: float(X[c].median()) for c in FEATURES}      # start from a typical order
my_order.update({"distance_km": 6.0, "prep_time_min": 15.0})  # change the numbers you like
row = pd.DataFrame([my_order])[FEATURES]
print(my_order)
print(f"Predicted delivery time: {best.predict(row)[0]:.1f} minutes")

## 10. Explanation (36-40 min) - auto-generated numbers, then edit the wording

In [ ]:
top = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
print(f'''
DATA PREPARATION
- Checked for missing values, duplicates and impossible values; outliers were kept unless clearly invalid.
- Dropped order_id (an identifier, no predictive value) and distance_miles (same as distance_km in other units; redundant, and it makes linear regression unstable).
- 80/20 train/test split with a fixed seed; the test set was used once, for the final score only.

MODEL CHOICE
- Baseline (guess the average): MAE {test_table["Guess the average"][0]:.2f} min.
- Compared linear regression, random forest and gradient boosting with 5-fold CV on the training set (MAE).
- Winner: {best_name} (CV MAE {cv_mae[best_name]:.2f} min).

RESULTS ON UNSEEN ORDERS
- MAE {mae:.2f} min, RMSE {rmse:.2f} min, R2 {r2:.3f}.
- Typical prediction is off by about {mae:.1f} minutes.
- Strongest drivers: {", ".join(top.index[:3])}.

LIMITATION
- traffic_delay_min and courier_wait_min are probably not known when an order is placed.
- Without them: MAE {mae2:.2f} min, R2 {r22:.3f} (vs {mae:.2f} / {r2:.3f} with them).
- In production we would use forecasts for those two inputs, or the order-time-only model.
''')